# Phase 2 market values explorer
Open a completed, certified Phase 2 V6 session. The full holding grid is time-major and can be large, so queries below are lazy and restricted to a short time window. `market_open_values.parquet` is sparse: a missing opening row means `can_open=False`. These are **hindsight labels**, not causal model inputs or executable trading returns. No cell writes to the source dataset.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import json
import polars as pl
from IPython.display import display

# Change this root to another certified Phase 2 date when needed.
REMOTE_ROOT = Path(r"\\DESKTOP-SAAI85T\Workstation-D\TradingML\runtimes\hindsight-greedy\2026-08-18\b5b2a7ddd83d676c")
LOCAL_ROOT = Path(r"D:\TradingML\runtimes\hindsight-greedy\2026-08-18\b5b2a7ddd83d676c")
PHASE2_ROOT = LOCAL_ROOT if LOCAL_ROOT.is_dir() else REMOTE_ROOT
TICKER = "AAPL"
SIDE = "long"  # or "short"
START_ET = "09:30:00"
WINDOW_MINUTES = 5
MAX_DISPLAY_ROWS = 30
assert SIDE in {"long", "short"}
assert 1 <= WINDOW_MINUTES <= 60, "Use a bounded window to avoid a full-market scan"
assert PHASE2_ROOT.is_dir(), f"Phase 2 root unavailable: {PHASE2_ROOT}"

In [ ]:
plan = json.loads((PHASE2_ROOT / "plan.json").read_text(encoding="utf-8"))
complete = json.loads((PHASE2_ROOT / "complete.json").read_text(encoding="utf-8"))
assert plan["version"] == "hindsight-greedy-fractional-v6"
assert complete["plan_hash"] == plan["plan_hash"]
tensor = complete["tensor"]
holding_path = PHASE2_ROOT / tensor["holding"]["file"]
opening_path = PHASE2_ROOT / tensor["opening"]["file"]
assert holding_path.is_file() and opening_path.is_file()
assert tensor["listing_count"] == len(plan["selected"])
print("Date:", plan["date"], "Listings:", tensor["listing_count"])
print("Holding rows:", tensor["holding"]["rows"], "Opening rows:", tensor["opening"]["rows"])
print("Discount policy:", plan["discount_policy"])
print("Liquidity filter:", plan["liquidity_filter"])
print("Holding schema:", pl.read_parquet_schema(holding_path))
print("Opening schema:", pl.read_parquet_schema(opening_path))
# This checks completion metadata and Parquet schemas; it does not rehash multi-GB files.

In [ ]:
matches = [(i, row) for i, row in enumerate(plan["selected"]) if row["ticker"] == TICKER]
assert len(matches) == 1, f"Expected one listing for {TICKER}, found {len(matches)}"
listing_index, listing = matches[0]
ny = ZoneInfo("America/New_York")
start_et = datetime.fromisoformat(f"{plan['date']}T{START_ET}").replace(tzinfo=ny)
end_et = start_et + timedelta(minutes=WINDOW_MINUTES)
start_us = int(start_et.timestamp() * 1_000_000)
end_us = int(end_et.timestamp() * 1_000_000)
print("Selected:", listing)
print("Window:", start_et, "to", end_et)
base_filter = ((pl.col("time_us") >= start_us) & (pl.col("time_us") < end_us) &
               (pl.col("listing_index") == listing_index) & (pl.col("side") == SIDE))
holding = pl.scan_parquet(holding_path).filter(base_filter)
opening = pl.scan_parquet(opening_path).filter(base_filter)
timeline = (holding.join(opening, on=["time_us", "listing_index", "side"], how="left")
    .with_columns(pl.col("can_open").fill_null(False),
                  pl.col("open_value_available").fill_null(False),
                  pl.from_epoch("time_us", time_unit="us").dt.replace_time_zone("UTC")
                    .dt.convert_time_zone("America/New_York").alias("time_et"))
    .sort("time_us").collect())
assert timeline.height == WINDOW_MINUTES * 60, "Expected one row per second for this listing and side"
print("Open-eligible seconds:", timeline["can_open"].sum())
display(timeline.select("time_et", "ticker", "side", "status", "can_open",
    "close_price", "entry_price", "target_price", "hold_seconds",
    "hold_profit_per_share", "hold_value_per_share",
    "open_profit_per_share", "open_value_per_dollar").head(MAX_DISPLAY_ROWS))

`hold_profit_per_share` and `open_profit_per_share` are undiscounted local hindsight outcomes. The corresponding `*_value_*` fields apply the Phase 2 discount. `hold_seconds` refers to the local hindsight target duration; it is **not** the duration of a Phase 3 portfolio position. An absent opening row does not invalidate the holding or closing row.

In [ ]:
import matplotlib.pyplot as plt
times = timeline["time_et"].to_list()
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(times, timeline["close_price"].to_list(), label="Current close reference")
axes[0].plot(times, timeline["target_price"].to_list(), label="Hindsight target", alpha=.7)
axes[0].set_ylabel("Price ($/share)")
axes[0].legend()
axes[1].plot(times, timeline["hold_value_per_share"].to_list(), label="Discounted hold value/share")
axes[1].plot(times, timeline["open_value_per_dollar"].to_list(), label="Discounted open value/$")
axes[1].set_ylabel("Hindsight value")
axes[1].legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
# One second at a time, across every selected listing and both sides.
# Change second_offset to 0, 1, 2, ... and rerun this cell.
second_offset = 0
assert second_offset >= 0
second_et = start_et + timedelta(seconds=second_offset)
second_us = int(second_et.timestamp() * 1_000_000)
holding_second = pl.scan_parquet(holding_path).filter(pl.col("time_us") == second_us)
opening_second = pl.scan_parquet(opening_path).filter(pl.col("time_us") == second_us)
second_table = (holding_second
    .join(opening_second, on=["time_us", "listing_index", "side"], how="left")
    .with_columns(pl.col("can_open").fill_null(False),
                  pl.col("open_value_available").fill_null(False))
    .sort("open_value_per_dollar", descending=True, nulls_last=True)
    .select("ticker", "listing_index", "side", "status", "can_close",
            "can_open", "entry_price", "close_price", "target_price",
            "hold_seconds", "hold_profit_per_share", "hold_value_per_share",
            "open_profit_per_share", "open_value_per_share",
            "open_value_per_dollar")
    .collect())
assert second_table.height == tensor["listing_count"] * 2
print("Completed second:", second_et, "| rows:", second_table.height,
      "| eligible openings:", second_table["can_open"].sum())
# second_table contains every listing/side. Null scores (no eligible opening) sort last.
# Polars truncates only the visual display; filter second_table to inspect any ticker.
display(second_table)

In [ ]:
# Optional: inspect the Phase 2 greedy projection for the same bounded window.
projection_path = PHASE2_ROOT / f"{SIDE}.parquet"
projection = (pl.scan_parquet(projection_path)
    .filter((pl.col("time_us") >= start_us) & (pl.col("time_us") < end_us))
    .sort("time_us").head(MAX_DISPLAY_ROWS).collect())
display(projection)
# This greedy projection is not the Phase 3 cash-constrained portfolio trajectory.